In [ ]:
import json
from pathlib import Path

traj_path = Path("e2e_trajectories_glm47.jsonl")
trajectories = [json.loads(line) for line in traj_path.read_text().strip().splitlines()]
print(f"Loaded {len(trajectories)} trajectories")

In [ ]:
from IPython.display import display, Markdown, HTML

for i, traj in enumerate(trajectories):
    msgs = traj["messages"]
    has_tool = any(m.get("tool_calls") for m in msgs)
    print(f"[{i}] query={traj['query'][:50]}  expect={traj['expect']}  msgs={len(msgs)}  tool_call={has_tool}")

In [ ]:
def show_trajectory(traj, idx=0):
    msgs = traj["messages"]
    display(Markdown(f"## Trajectory {idx}: {traj['query']}"))
    display(Markdown(f"**Expect:** `{traj['expect']}`\n\n---"))

    for m in msgs:
        role = m["role"]

        if role == "system":
            display(Markdown(f"### 🔧 System\n```\n{m['content'][:300]}...\n```"))

        elif role == "user":
            display(Markdown(f"### 👤 User\n> {m['content']}"))

        elif role == "assistant" and m.get("tool_calls"):
            tc = m["tool_calls"][0]
            fn = tc["function"]
            args = json.loads(fn["arguments"]) if isinstance(fn["arguments"], str) else fn["arguments"]
            args_pretty = json.dumps(args, indent=2, ensure_ascii=False)
            text = m.get("content") or ""
            display(Markdown(
                f"### 🤖 Assistant (tool call)\n"
                f"{text}\n\n"
                f"**Call:** `{fn['name']}`\n```json\n{args_pretty}\n```"
            ))

        elif role == "tool":
            content = m["content"]
            preview = content[:500] + ("..." if len(content) > 500 else "")
            display(Markdown(f"### 🔨 Tool Result ({len(content)} chars)\n```\n{preview}\n```"))

        elif role == "assistant":
            display(Markdown(f"### 🤖 Assistant (final)\n{m['content']}"))

    display(Markdown("---\n"))

In [ ]:
show_trajectory(trajectories[0], 0)

In [ ]:
for i, traj in enumerate(trajectories):
    show_trajectory(traj, i)